# Part 1

Đây là file notebook được sử dụng để kiểm chứng các hàm đã được cài đặt trong `part1`. 


Các hàm sẽ được kiểm chứng bao gồm:
- `gaussian_eliminate(A, b)`
- `back_substitution(U, c)`
- `determinant(A)`
- `inverse(A)`
- `rank_and_basis(A)`


## Import các hàm 

In [1]:
import numpy as np
from gaussian import gaussian_eliminate
from back_substitution import Expression, back_substitution
from determinant import determinant
from inverse import inverse
from rank_basis import rank_and_basis
from utils import verify_solution

## Kiểm nghiệm `gaussian_eliminate` 

## Kiểm nghiệm kết quả

Sử dụng hàm `verify_solution(A, x, b)` để kiểm nghiệm các hàm đã được cài đặt trong folder `part1/`

In [ ]:
def _to_float(v):
    if isinstance(v, Expression):
        if any(k != Expression.BIAS for k in v.mp):
            raise ValueError("Symbolic expression")
        return float(v.mp.get(Expression.BIAS, 0.0))
    return float(v)

def _to_np(mat):
    if mat is None: raise ValueError
    if isinstance(mat, np.ndarray): return mat.astype(float)
    if not mat: return np.empty((0, 0), dtype=float)
    if isinstance(mat[0], (list, tuple)):
        return np.array([[_to_float(v) for v in row] for row in mat], dtype=float)
    return np.array([_to_float(v) for v in mat], dtype=float)

def verify_solution(A, x, b):
    """Verify determinant, inverse, rank, and gaussian_eliminate against NumPy."""

    A_np = _to_np(A)
    n, m = A_np.shape
    results = {}

    # Check determinant & inverse (square only)
    if n == m:
        try:
            results["det_ok"] = np.isclose(determinant(A), np.linalg.det(A_np), atol=1e-8, rtol=1e-6)
        except: results["det_ok"] = False
        try:
            inv = inverse(A)
            results["inv_ok"] = (np.linalg.matrix_rank(A_np) < n) if inv is None else np.allclose(_to_np(inv), np.linalg.inv(A_np), atol=1e-8)
        except: results["inv_ok"] = False

    # Check rank
    try:
        r, _, _, _ = rank_and_basis(A)
        results["rank_ok"] = (r == np.linalg.matrix_rank(A_np))
    except: results["rank_ok"] = False

    # Check gaussian_eliminate
    try:
        _, x_calc, _ = gaussian_eliminate(A, b)
        x_np = _to_np(x_calc).reshape(-1, 1) if _to_np(x_calc).ndim == 1 else _to_np(x_calc)
        b_np = _to_np(b).reshape(-1, 1) if _to_np(b).ndim == 1 else _to_np(b)
        results["gauss_ok"] = np.allclose(A_np @ x_np, b_np, atol=1e-8)
    except: results["gauss_ok"] = None

    return results


def print_matrix(mat):
    for row in mat:
        print(["{:.4f}".format(v) for v in row])


def gaussian_eliminate_formatter(U, x, num_swaps):
    """
    Formats the gaussian_eliminate() result
    """
    
    # print upper triangle matrix
    print("\nUpper Triangle Matrix U:")
    print("-" * 40)
    if U:
        for i, row in enumerate(U):
            row_str = "["
            for j, val in enumerate(row):
                if isinstance(val, float):
                    row_str += f"{val:10.4f}"
                else:
                    row_str += f"{str(val):>10}"
                if j < len(row) - 1:
                    row_str += ", "
            row_str += "]"
            print(row_str)
    else:
        print("Empty matrix")
    
    # print solution vector
    print("\nSolution Vector x:")
    print("-" * 40)
    if x:
        for i, val in enumerate(x):
            print(f"x[{i}] = {val}")
    else:
        print("No solution")
    
    # print number of swaps
    print(f"\nNumber of Row Swaps: {num_swaps}")
